In [1]:
import time
import pandas as pd
from selenium import webdriver
from selenium.webdriver import ActionChains
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.edge.service import Service
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.common.exceptions import NoSuchElementException, StaleElementReferenceException

In [2]:
def strip_parent(string):
    return string.split('(')[1]

def switch_date(driver, go_to_date):
    '''
    go_to_date[int]: 要切换到的日期
    '''
    date_xpath = '/html/body/div[1]/div/main/div/div[2]/div[1]/div[1]/div[1]/div[2]/div/div/div/div[2]/button[{num_date}]/div/span'.format(num_date=str(go_to_date))
    date_element = driver.find_element(By.XPATH, date_xpath)
    date = date_element.text
    date_element.click()
    print('已切换至',date,'日')
    
def get_match(driver):
    '''
    获取完场比赛比分及盘口信息
    '''
    #打开新标签页获取盘口
    element = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/a/button'))) #Show More Button
    element.send_keys(Keys.CONTROL + Keys.RETURN)
    driver.switch_to.window(driver.window_handles[1])
    time.sleep(3)
    
    #获取比分及盘口
    try:
        driver.find_element(By.CLASS_NAME,'sc-eDWCr.jgSsSj') #主比分1
        homescore_class_name = 'sc-eDWCr.jgSsSj'
    except NoSuchElementException:
        driver.find_element(By.CLASS_NAME,'sc-eDWCr.iRYKkj') #主比分2
        homescore_class_name = 'sc-eDWCr.iRYKkj'
    except NoSuchElementException:
        print('暂无比分')
    try:
        driver.find_element(By.CLASS_NAME,'sc-eDWCr.hNUSos') #客比分1
        awayscore_class_name = 'sc-eDWCr.hNUSos'
    except NoSuchElementException:
        driver.find_element(By.CLASS_NAME,'sc-eDWCr.eRrOEk') #客比分2
        awayscore_class_name = 'sc-eDWCr.eRrOEk'
    except NoSuchElementException:
        print('暂无比分')
    
    home_score = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME, homescore_class_name))).text
    away_score = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME, awayscore_class_name))).text
    test_flags = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.CLASS_NAME,'sc-eDWCr.itJafI')))
    handicap_H = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.itJafI')[0].text #主盘口
    handicap_A = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.itJafI')[1].text #客盘口
    value_home = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.dsMMht')[2].text #主赔
    value_away = driver.find_elements(By.CLASS_NAME,'sc-eDWCr.dsMMht')[3].text #客赔    
    
    #回到主页面
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    return [home_score, away_score, handicap_H, handicap_A, value_home, value_away]

def clean_handicap(handicap_value, value_home, value_away):
    '''
    清洗盘口信息
    '''
    if handicap_value == '0' or handicap_value == '-0':
        if value_home < value_away:
            handicap_value = '-0'
        elif value_away < value_home:
            handicap_value = '+0'
        else:
            handicap_value = '0'
    else:
        if not handicap_value.startswith('-'):
            handicap_value = '+'+handicap_value
    return handicap_value

def get_class_name(driver, flag):
    '''
    动态切换class_name
    '''
    CLASS_NAME = ''
    if flag == 'matches':
        try:
            driver.find_element(By.CLASS_NAME,'sc-hKwDye.LFQYO.sc-9199a964-1.bnnDyH') #比赛列表1
            CLASS_NAME = 'sc-hKwDye.LFQYO.sc-9199a964-1.bnnDyH'
        except NoSuchElementException:
            driver.find_element(By.CLASS_NAME,'sc-hLBbgP.dRtNhU.sc-9199a964-1.kusmLq') #比赛列表2
            CLASS_NAME = 'sc-hLBbgP.dRtNhU.sc-9199a964-1.kusmLq'
            
    elif flag == 'home_score':
        try:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.jgSsSj') #主比分1
            CLASS_NAME = 'sc-eDWCr.jgSsSj'
        except NoSuchElementException:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.iRYKkj') #主比分2
            CLASS_NAME = 'sc-eDWCr.iRYKkj'
            
    elif flag == 'away_score':
        try:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.hNUSos') #客比分1
            CLASS_NAME = 'sc-eDWCr.hNUSos'
        except NoSuchElementException:
            driver.find_element(By.CLASS_NAME,'sc-eDWCr.eRrOEk') #客比分2
            CLASS_NAME = 'sc-eDWCr.eRrOEk'
            
    return CLASS_NAME

def init_class_name(driver):
    '''
    初始化class_name
    '''
    match_class_name = get_class_name(driver, 'matches')
    driver.find_element(By.CLASS_NAME, match_class_name).click()
    
    element = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[3]/div/div[1]/div/a/button'))) #Show More Button
    element.send_keys(Keys.CONTROL + Keys.RETURN)
    driver.switch_to.window(driver.window_handles[1])
    time.sleep(15)
    homescore_class_name = get_class_name(driver, 'home_score')
    awayscore_class_name = get_class_name(driver, 'away_score')
    
    driver.close()
    driver.switch_to.window(driver.window_handles[0])
    
    return match_class_name, homescore_class_name, awayscore_class_name

In [22]:
driver_path = r'D:\edgedriver_win64\msedgedriver.exe' #PC
#driver_path = r'D:\EdgeDriver\msedgedriver.exe' #company
driver = webdriver.Edge(service=Service(executable_path=driver_path))
driver.implicitly_wait(10)
driver.get("https://www.sofascore.com/")

#显示赔率
showodds = driver.find_element(By.CLASS_NAME,'slider')
showodds.click()
print('初始化完成！')

初始化完成！


In [6]:
#模拟登录
profile = driver.find_element(By.XPATH,'/html/body/div[1]/div/header/div[1]/div/div[5]/div/a[3]')
profile.click()
time.sleep(5)

#Google登录
continue_with_google = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH, '/html/body/div[1]/div/main/div/div[2]/div/div/button[2]')))
continue_with_google.click()
for handle in driver.window_handles:
    driver.switch_to.window(handle)

#email
email = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.ID, "identifierId")))
email.send_keys('judd147@alumni.wfu.edu')
next_step = driver.find_element(By.XPATH,'/html/body/div[1]/div[1]/div[2]/div/div[2]/div/div/div[2]/div/div[2]/div/div[1]/div/div/button/span')
next_step.click()

#password
key = input('请输入密码:')
password = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.NAME, 'password')))
password.send_keys(key)
next_step = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div[1]/div[2]/div/div[2]/div/div/div[2]/div/div[2]/div/div[1]/div/div/button/span')))
next_step.click()

time.sleep(10)

#回到首页
driver.switch_to.window(driver.window_handles[0])
football = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/header/div[2]/div/div/div[1]/ul[2]/li[1]/a')))
football.click()

expand = WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[2]/div/div[2]/div')))
expand.click()
print('登录成功')

请输入密码:judd147t
登录成功


In [23]:
driver.find_element(By.XPATH,'/html/body/div[1]/div/main/div/div[2]/div[1]/div[1]/div[1]/div[1]/button[1]/button').click() #返回上个月
switch_date(driver, 30) #切换日期

已切换至 30 日


In [18]:
driver.close()
driver.switch_to.window(driver.window_handles[0])

In [24]:
#数据存储
#FIXME 1.如何根据数据库缺盘口的比赛寻找sofascore比赛
df_result = pd.DataFrame(columns=['主队','客队','H','A','盘口'])
first_scroll_flag = True
pinned_match_flag = True
amount_scrolled = 0
while pinned_match_flag:
    i = 0 #比赛index
    num_clicks = 0 #点击比赛次数
    driver.refresh() #刷新页面
    ActionChains(driver).scroll_by_amount(0, amount_scrolled).perform()
    time.sleep(5) #等待页面元素刷新
    
    try:
        driver.find_element(By.CLASS_NAME,'sc-hKwDye.LFQYO.sc-9199a964-1.bnnDyH') #比赛列表1
        match_class_name = 'sc-hKwDye.LFQYO.sc-9199a964-1.bnnDyH'
    except NoSuchElementException:
        driver.find_element(By.CLASS_NAME,'sc-hLBbgP.dRtNhU.sc-9199a964-1.kusmLq') #比赛列表2
        match_class_name = 'sc-hLBbgP.dRtNhU.sc-9199a964-1.kusmLq'
    matches = driver.find_elements(By.CLASS_NAME, match_class_name) #比赛列表
    
    #container = driver.find_element(By.ID,'pinned-list-fade-target') #收藏夹
    #item_list = container.text.split('\n') #收藏夹当前显示的比赛列表
    
    while num_clicks < 6:
        match = matches[i]
        i += 1
        match.click()
        num_clicks += 1
    
        info_list = get_match(driver)
        home_score = info_list[0]
        away_score = info_list[1]
        handicap_H = info_list[2]
        handicap_A = info_list[3]
        value_home = info_list[4]
        value_away = info_list[5]

        #显示信息
        try:
            home_name = handicap_H.split(') ')[1]
            away_name = handicap_A.split(') ')[1]
            handicap_value = strip_parent(handicap_H.split(') ')[0])
            handicap_value = clean_handicap(handicap_value, value_home, value_away)
        except:
            print('error parsing handicap info')

        print(home_name+' '+home_score+'-'+away_score+' '+away_name)
        print('盘口：', handicap_value)
        df_result = df_result.append({'主队':home_name, '客队':away_name, 'H':home_score, 'A':away_score, '盘口':handicap_value}, ignore_index=True)
        
        #判断结束条件
        #if home_name not in item_list:
        #    pinned_match_flag = False

    #划动比赛
    if first_scroll_flag:
        ActionChains(driver).scroll_by_amount(0, 700).perform()
        amount_scrolled += 700
        first_scroll_flag = False
    else:
        ActionChains(driver).scroll_by_amount(0, 350).perform()
        amount_scrolled += 350
    print('已划动')
    del matches

Flamengo 1-0 Athletico
盘口： -1
Flamengo 1-0 Athletico
盘口： -1
Fulham 0-0 Everton
盘口： -0.25
Liverpool 1-2 Leeds United
盘口： -2
Arsenal 5-0 Nottingham Forest
盘口： -1.75
Sevilla 0-1 Rayo Vallecano
盘口： -0.5
已划动
Sevilla 0-1 Rayo Vallecano
盘口： -0.5
Sevilla 0-1 Rayo Vallecano
盘口： -0.5
Valencia 0-1 Barcelona
盘口： +1
Osasuna 2-0 Real Valladolid
盘口： -0.5
Real Madrid 1-1 Girona FC
盘口： -1.75
Eintracht Frankfurt 1-2 Borussia Dortmund
盘口： -0
已划动
Lecce 0-1 Juventus
盘口： +0.25
Lecce 0-1 Juventus
盘口： +0.25
Inter 3-0 Sampdoria
盘口： -2
Empoli 0-2 Atalanta
盘口： +0.75
Cremonese 0-0 Udinese
盘口： +0.5
Spezia 1-2 Fiorentina
盘口： +0.5
已划动
RC Strasbourg 2-2 Olympique de Marseille
盘口： +0.25
RC Strasbourg 2-2 Olympique de Marseille
盘口： +0.25
AJ Auxerre 1-0 AC Ajaccio
盘口： -0
AS Monaco 2-0 Angers
盘口： -1.25
FC Nantes 1-1 Clermont Foot 63
盘口： -0.5
Stade Brestois 29 0-0 Stade de Reims
盘口： +0
已划动
SL Benfica 5-0 GD Chaves
盘口： -2.5
SL Benfica 5-0 GD Chaves
盘口： -2.5
Arouca 1-0 Sporting CP
盘口： +0.75
Boavista 2-2 FC Vizela
盘口： +0
Por

KeyboardInterrupt: 

In [ ]:
driver.quit()

In [98]:
df_result29.drop_duplicates(subset=['比赛'], inplace=True)
df_result29

,比赛,盘口
0,Leicester City 0-1 Manchester City,+1.25
1,AFC Bournemouth 2-3 Tottenham Hotspur,+0.75
2,Brentford 1-1 Wolverhampton,-0.25
3,Brighton & Hove Albion 4-1 Chelsea,+0.25
4,Crystal Palace 1-0 Southampton,-0.5
5,Newcastle United 4-0 Aston Villa,-0.75
6,RCD Mallorca 1-1 Espanyol,-0.25
7,UD Almería 3-1 Celta Vigo,+0.25
8,Cádiz CF 3-2 Atlético Madrid,+0.75
9,Werder Bremen 1-0 Hertha BSC,-0.25


In [25]:
df_result30 = df_result.drop_duplicates(subset=['主队','客队'])
df_result30.to_excel(r'C:\Users\张力铫\Downloads\xingchenscrapers-main\result1030.xlsx', index=False)
df_result30

,主队,客队,H,A,盘口
0,Flamengo,Athletico,1,0,-1
2,Fulham,Everton,0,0,-0.25
3,Liverpool,Leeds United,1,2,-2
4,Arsenal,Nottingham Forest,5,0,-1.75
5,Sevilla,Rayo Vallecano,0,1,-0.5
8,Valencia,Barcelona,0,1,+1
9,Osasuna,Real Valladolid,2,0,-0.5
10,Real Madrid,Girona FC,1,1,-1.75
11,Eintracht Frankfurt,Borussia Dortmund,1,2,-0
12,Lecce,Juventus,0,1,+0.25


In [20]:
df_result31 = df_result.drop_duplicates(subset=['主队','客队'])
df_result31.to_excel(r'C:\Users\张力铫\Downloads\xingchenscrapers-main\result1031.xlsx', index=False)
df_result31

,主队,客队,H,A,盘口
0,Manchester United,West Ham United,1,0,-0.75
2,Athletic Club,Villarreal,1,0,-0.25
3,Real Sociedad,Real Betis Balompié,0,2,-0.5
4,FC Schalke 04,SC Freiburg,0,2,+0.5
5,1. FC Köln,1899 Hoffenheim,1,1,+0.25
8,Lazio,Salernitana,1,3,-1
9,Torino,Milan,2,1,+0.5
10,Lorient,OGC Nice,1,2,-0
11,Olympique Lyonnais,Lille OSC,1,0,-0.5
12,Casa Pia AC,Rio Ave,1,0,-0.5


In [25]:
df_result01 = df_result.drop_duplicates(subset=['主队','客队'])
df_result01.to_excel(r'C:\Users\张力铫\Downloads\xingchenscrapers-main\result1101.xlsx', index=False)
df_result01

,主队,客队,H,A,盘口
0,Elche CF,Getafe CF,0,1,-0
2,Hellas Verona,Roma,1,3,+0.5
3,Monza,Bologna,1,2,-0.25
4,Vitória SC,FC Famalicão,3,2,-0.25
5,Başakşehir FK,Giresunspor,3,1,-0.75
7,Sivasspor,Antalyaspor,0,2,-0.25
9,Ceará,Fluminense,0,1,+0.25
10,Levante UD,Sporting Gijón,1,0,-0.75
11,Real Zaragoza,FC Andorra,0,2,-0.25
15,SD Ponferradina,Huesca,1,0,-0


In [11]:
df_result02 = df_result.drop_duplicates(subset=['主队','客队'])
df_result02.to_excel(r'C:\Users\张力铫\Downloads\xingchenscrapers-main\result1102.xlsx', index=False)
df_result02

,主队,客队,H,A,盘口
0,Liverpool,Napoli,2,0,-0.5
2,Rangers,Ajax,1,3,+0.5
3,Bayer 04 Leverkusen,Club Brugge,0,0,-1
4,FC Porto,Atlético Madrid,2,1,+0
5,Bayern München,Inter,2,0,-1.25
8,Viktoria Plzeň,Barcelona,2,4,+1.5
9,Olympique de Marseille,Tottenham Hotspur,1,2,+0.25
10,Sporting CP,Eintracht Frankfurt,1,2,-0.25
11,Botafogo,Cuiabá,0,2,-0.5
14,São Paulo,Atlético Mineiro,2,2,-0.25


In [ ]:
#球队字典
#FIXME UPDATE
def clean_teams(home, away, league_name):
    '''
    返回清洗后的球队名称
    '''
    teams_dict = {'日乙':{'群马温泉':'群马草津温泉','金泽塞维根':'金泽','琉球FC':'FC琉球'},
                  '美职联':{'辛辛那提FC':'辛辛那提','温哥华白帽':'温哥华白浪','堪萨斯城竞技':'堪萨斯城体育','波特兰伐木工':'波特兰伐木者'},
                  '日职联':{'鸟栖沙岩':'鸟栖砂岩','清水鼓动':'清水心跳','名古屋鲸八':'名古屋逆戟鲸'},
                  '阿甲':{'普拉腾斯':'普拉滕斯竞技','泰格雷':'老虎竞技','竞技俱乐部':'竞技','圣塔菲联':'圣菲联','巴拉卡斯中央队':'巴拉卡斯中央',
                               '科隆竞技':'哥伦布竞技','铁路工场':'塔列雷斯','阿尔多西维':'阿尔多希维','科尔多瓦中央SDE':'科尔多瓦中央',
                               '阿根廷独立':'独立','萨尔米安杜':'萨米恩托','飓风队':'飓风','帕特罗纳图':'天主教青年','防御与正义':'国防与司法'},
                  #✔
                  '德甲':{'Werder Bremen':'云达不莱梅','Hertha BSC':'柏林赫塔','Bayern München':'拜仁慕尼黑','1. FSV Mainz 05':'美因茨','VfL Bochum':'波鸿',
                           'RB Leipzig':'RB莱比锡','Bayer 04 Leverkusen':'勒沃库森','VfB Stuttgart':'斯图加特','FC Augsburg':'奥格斯堡',
                           'VfL Wolfsburg':'沃尔夫斯堡','Eintracht Frankfurt':'法兰克福','Borussia Dortmund':'多特蒙德','FC Schalke 04':'沙尔克04',
                           'SC Freiburg':'弗赖堡','1. FC Köln':'科隆','1899 Hoffenheim':'霍芬海姆','1. FC Union Berlin':'柏林联合',"Borussia M'gladbach":'门兴格拉德巴赫',},
                  #✔
                  '西甲':{'Celta Vigo':'塞尔塔','RCD Mallorca':'马略卡','UD Almería':'阿尔梅里亚','Real Valladolid':'巴拉多利德',
                           'Cádiz CF':'加迪斯','Espanyol':'西班牙人','Atlético Madrid':'马德里竞技','Sevilla':'塞维利亚',
                           'Rayo Vallecano':'巴列卡诺','Valencia':'巴伦西亚','Barcelona':'巴塞罗那','Osasuna':'奥萨苏纳',
                           'Real Madrid':'皇家马德里','Girona FC':'赫罗纳','Athletic Club':'毕尔巴鄂竞技','Villarreal':'比利亚雷亚尔',
                           'Real Sociedad':'皇家社会','Real Betis Balompié':'皇家贝蒂斯','Elche CF':'埃尔切','Getafe CF':'赫塔费'},
                  #✔
                  '英超':{'Leicester City':'莱斯特城','Manchester City':'曼城','AFC Bournemouth':'伯恩茅斯','Tottenham Hotspur':'热刺',
                           'Brentford':'布伦特福德','Wolverhampton':'狼队','Brighton & Hove Albion':'布莱顿','Chelsea':'切尔西',
                           'Crystal Palace':'水晶宫','Southampton':'南安普顿','Newcastle United':'纽卡斯尔','Aston Villa':'阿斯顿维拉',
                           'Liverpool':'利物浦','Leeds United':'利兹联','Arsenal':'阿森纳','Nottingham Forest':'诺丁汉森林',
                           'Manchester United':'曼联','West Ham United':'西汉姆联','Fulham':'富勒姆','Everton':'埃弗顿'},
                  #✔
                  '法甲':{'RC Lens':'朗斯','Toulouse':'图卢兹','Paris Saint-Germain':'巴黎圣日尔曼','Troyes':'特鲁瓦',
                           'RC Strasbourg':'斯特拉斯堡','Olympique de Marseille':'马赛','AJ Auxerre':'欧塞尔','AC Ajaccio':'阿雅克肖',
                           'AS Monaco':'摩纳哥','Angers':'昂热','FC Nantes':'南特','Clermont Foot 63':'克莱蒙',
                           'Stade Brestois 29':'布雷斯特','Stade de Reims':'兰斯','Lorient':'洛里昂','OGC Nice':'尼斯',
                           'Olympique Lyonnais':'里昂','Lille OSC':'里尔','Stade Rennais':'雷恩','Montpellier':'蒙彼利埃'},
                  #✔
                  '意甲':{'Napoli':'那不勒斯','Sassuolo':'萨索洛','Lecce':'莱切','Juventus':'尤文图斯','Udinese':'乌迪内斯',
                           'Inter':'国际米兰','Sampdoria':'桑普多利亚','Empoli':'恩波利','Atalanta':'亚特兰大','Cremonese':'克雷莫内塞',
                           'Spezia':'斯佩齐亚','Fiorentina':'佛罗伦萨','Lazio':'拉齐奥','Salernitana':'萨勒尼塔纳','Torino':'都灵',
                           'Milan':'AC米兰','Hellas Verona':'维罗纳','Roma':'罗马','Monza':'蒙扎','Bologna':'博洛尼亚'},
                  '欧冠':{'比尔森':'比尔森胜利','莱比锡红牛':'RB莱比锡',
                           '萨尔茨堡':'萨尔茨堡红牛','格拉斯哥流浪者':'流浪者'},
                  '欧联':{'谢里夫':'蒂拉斯波尔警长',},
                  '欧协联':{},#FIXME
                  '德乙':{'不伦瑞克':'布伦瑞克'},
                  '英冠':{'加的夫城':'卡迪夫城','布里斯托城':'布里斯托尔城','西布罗姆维奇':'西布朗'},
                  '西乙':{'格拉纳达GF':'格拉纳达','米兰迪斯':'米兰德斯','安道尔CF':'FC安道尔','阿尔巴切特':'阿尔瓦塞特','特內里费':'特内里费'},
                  '巴甲':{'奥瓦':'阿瓦伊','科里蒂巴':'库里蒂巴','布拉干蒂诺RB':'布拉甘蒂诺红牛','戈伊亚斯':'戈亚斯','福塔雷萨':'福塔莱萨','库亚巴':'奎尔巴'},
                  '墨超':{'老虎大学':'墨西哥老虎','马萨特兰FC':'马萨特兰','蒙特瑞':'蒙特雷','墨西哥美洲':'美洲','阿苏尔':'蓝十字',
                           '拿加沙':'内卡萨','圣路易斯竞技':'圣路易斯'},
                  #✔
                  '葡超':{'Portimonense':'波尔蒂芒人','GD Chaves':'沙维斯','Gil Vicente FC':'吉尔维森特','Rio Ave':'阿维河','Sporting CP':'葡萄牙体育',
                           'Casa Pia AC':'卡萨皮亚','FC Vizela':'维泽拉','Paços de Ferreira':'帕索斯费雷拉','CS Marítimo':'马德拉航海',
                           'CD Santa Clara':'圣克拉拉','FC Porto':'波尔图','SL Benfica':'本菲卡','Arouca':'阿罗卡','FC Famalicão':'法马利康',
                           'Boavista':'博阿维斯塔','Estoril Praia':'埃斯托里尔','Sporting Braga':'布拉加','Vitória SC':'吉马良斯'},
                  #✔
                  '荷甲':{'FC Emmen':'埃蒙','Fortuna Sittard':'锡塔德幸运','Vitesse':'维特斯','NEC Nijmegen':'奈梅亨','Feyenoord':'费耶诺德',
                           'SC Heerenveen':'海伦芬','FC Utrecht':'乌德勒支','Sparta Rotterdam':'鹿特丹斯巴达','FC Groningen':'格罗宁根',
                           'FC Twente':'特温特','RKC Waalwijk':'瓦尔韦克','PSV Eindhoven':'埃因霍温','AZ Alkmaar':'阿尔克马尔',
                           'FC Volendam':'福伦丹','Go Ahead Eagles':'前进之鹰','Excelsior':'SBV精英','Ajax':'阿贾克斯','SC Cambuur':'坎布尔'},
                  '瑞典超':{'韦纳穆':'瓦纳默','IFK哥德堡':'哥德堡','AIK索尔纳':'索尔纳'},
                  '挪超':{'奥德':'奥特','博德闪耀':'博多格林特','格里姆斯塔':'谢夫','桑纳菲尤尔':'桑德菲杰','萨尔普斯堡':'萨普斯堡'},
                  '比甲':{'奥德赫维里':'奥哈瓦里','聚尔特瓦雷赫姆':'威尔郡','沙勒罗瓦':'沙勒鲁瓦','瑟兰联':'塞莱恩'},
                  '智甲':{'尤尼昂':'拉卡勒拉联','科金博':'科金博联合','维尼亚德马埃弗顿':'比尼亚德尔马埃弗顿','库里科':'库里科联合',
                           '塞雷那':'拉塞雷纳','奴伯伦斯':'纽布伦斯','希金斯':'奥希金斯','科布雷索':'科布雷萨尔','奥达斯':'奥达科斯意大利人',
                           '华奇巴托':'瓦奇巴托'},
                  '亚冠':{},#FIXME
                  '解放者杯':{},#FIXME
                  '南球杯':{},#FIXME
                  '中超':{} #FIXME
                  }
    if league_name in teams_dict:
        for key, value in teams_dict[league_name].items():
            home = home.replace(key, value)
            away = away.replace(key, value)
    return home, away